# Graphs

**Geometry type:** `graph`

Vertices at positions joined by edges, with cycles allowed: vascular
networks, connectivity graphs. The whole graph is one object.

1. Generate a synthetic vascular network
2. Write it with vertex and edge attributes
3. Open the store and inspect its metadata
4. Read all vertices and edges
5. Bounding-box query
6. Validate
7. Convert a graph from CSV files

In [1]:
import os
import tempfile

import numpy as np
import zarr_vectors as zv
from zarr_vectors.types.graphs import read_graph, write_graph
from zarr_vectors.validate import validate

_tmpdir = tempfile.mkdtemp(prefix="zv_examples_")
STORE = os.path.join(_tmpdir, "vessels.zv")
print("Store path:", STORE)

Store path: /tmp/zv_examples_2334bw3m/vessels.zv


## 1 · Generate a synthetic vascular network

3,000 vertices in a 600 µm cube, each joined to its four nearest neighbours.

In [2]:
from scipy.spatial import KDTree

rng = np.random.default_rng(2)
n_nodes = 3_000

positions = rng.uniform(0, 600, (n_nodes, 3)).astype(np.float32)   # µm

_, nbrs = KDTree(positions).query(positions, k=5)   # the first neighbour is the vertex itself
src = np.repeat(np.arange(n_nodes, dtype=np.int32), 4)
dst = nbrs[:, 1:].ravel().astype(np.int32)

# Undirected: sort each pair and drop repeats.
edges = np.unique(np.sort(np.column_stack([src, dst]), axis=1), axis=0).astype(np.int32)

diameter = rng.uniform(1, 20, n_nodes).astype(np.float32)        # per vertex, µm
flow     = rng.uniform(0, 1, n_nodes).astype(np.float32)         # per vertex
edge_len = np.linalg.norm(positions[edges[:, 0]] - positions[edges[:, 1]], axis=1).astype(np.float32)

print(f"vertices    : {n_nodes:,}")
print(f"edges       : {len(edges):,}")
print(f"mean degree : {2 * len(edges) / n_nodes:.1f}")

vertices    : 3,000
edges       : 7,428
mean degree : 5.0


## 2 · Write the graph

Edges whose two vertices fall in different chunks are stored as cross-chunk
links; the summary counts both kinds.

In [3]:
summary = write_graph(
    STORE,
    positions=positions,
    edges=edges,
    chunk_shape=(150.0, 150.0, 150.0),
    bin_shape=(37.5, 37.5, 37.5),
    kind="graph",
    vertex_attributes={"diameter": diameter, "flow": flow},
    link_attributes={"length": edge_len},
)
print(summary)

{'node_count': 3000, 'edge_count': 7428, 'chunk_count': 64, 'intra_edge_count': 5711, 'cross_edge_count': 1717, 'object_count': 1, 'kind': 'graph'}


## 3 · Open and inspect

In [4]:
ds = zv.open(STORE)
lv0 = ds.level(0)
print(ds)
print(f"chunk shape  : {lv0.scale} µm")
print(f"vertex_count : {lv0.vertex_count:,}")
print(f"objects      : {lv0.objects.count}")
print(f"vertex attrs : {lv0.attribute_names('vertex')}")
print(f"link attrs   : {lv0.attribute_names('link')}")
print(f"bounds       : {ds.bounds[0].round(1)} → {ds.bounds[1].round(1)} µm")

Dataset('file:///tmp/zv_examples_2334bw3m/vessels.zv', graph, levels=[0])
chunk shape  : (150.0, 150.0, 150.0) µm
vertex_count : 3,000
objects      : 1
vertex attrs : ('diameter', 'flow')
link attrs   : ('length',)
bounds       : [0.  0.  0.8] → [599.8 599.9 600. ] µm


## 4 · Read all vertices and edges

`read_graph` returns vertices in store order (chunk by chunk), not input
order, with `edges` numbered in that order. `ds.read()` returns the same
arrays plus the vertex attributes.

In [5]:
result = read_graph(STORE)
print(f"node_count : {result['node_count']:,}")
print(f"edge_count : {result['edge_count']:,}")
print(f"positions  : {result['positions'].shape}")
print(f"edges      : {result['edges'].shape}")

# Map each stored vertex back to its input row and compare the edge sets.
input_row = {tuple(p): i for i, p in enumerate(positions)}
to_input = np.array([input_row[tuple(p)] for p in result["positions"]])
stored = {tuple(sorted(pair)) for pair in to_input[result["edges"]].tolist()}
print(f"same edges as the input: {stored == set(map(tuple, edges.tolist()))}")

full = ds.read()
print(f"\nds.read(): {full.vertex_count:,} vertices, {len(full.edges):,} edges, "
      f"attributes {full.attributes.names()}")
print(f"diameter matches input: {np.allclose(full.attributes['diameter'], diameter[to_input])}")

node_count : 3,000
edge_count : 7,428
positions  : (3000, 3)
edges      : (7428, 2)
same edges as the input: True



ds.read(): 3,000 vertices, 7,428 edges, attributes ('diameter', 'flow')
diameter matches input: True


## 5 · Bounding-box query

A bounding-box read returns the vertices inside the box and the edges with both
ends inside, renumbered within the result.

In [6]:
lo = np.array([200.0, 200.0, 200.0])
hi = np.array([400.0, 400.0, 400.0])

bbox_result = read_graph(STORE, bbox=(lo, hi))
print(f"vertices in the 200 µm box : {bbox_result['node_count']:,}")
print(f"edges with both ends inside: {bbox_result['edge_count']:,}")

inside = ((positions >= lo) & (positions <= hi)).all(axis=1)
print(f"input: {inside.sum()} vertices, {(inside[edges[:, 0]] & inside[edges[:, 1]]).sum()} edges")

vertices in the 200 µm box : 98
edges with both ends inside: 173
input: 98 vertices, 173 edges


## 6 · Validate

In [7]:
print(validate(STORE, level=3).summary())

Level 3 validation: PASS
  25 passed, 0 warnings, 0 errors


## 7 · Convert a graph from CSV files

A graph often starts as two tables: vertices with an id and a position, and
edges naming two vertex ids. `ingest_edgelist` reads them; every other column
becomes a vertex or edge attribute (float32). `compute_degree=True` adds each
vertex's degree (needs networkx, the `[graph]` extra).

In [8]:
import pandas as pd
from zarr_vectors_tools.convert.ingest.edgelist import ingest_edgelist

nodes_csv = os.path.join(_tmpdir, "nodes.csv")
edges_csv = os.path.join(_tmpdir, "edges.csv")
pd.DataFrame({
    "node_id": np.arange(n_nodes) + 1000,          # ids need not be row numbers
    "x": positions[:, 0], "y": positions[:, 1], "z": positions[:, 2],
    "diameter": diameter,
}).to_csv(nodes_csv, index=False)
pd.DataFrame({
    "source": edges[:, 0] + 1000, "target": edges[:, 1] + 1000, "length": edge_len,
}).to_csv(edges_csv, index=False)
print(open(nodes_csv).readline().strip(), "|", open(edges_csv).readline().strip())

CSV_STORE = os.path.join(_tmpdir, "vessels_from_csv.zv")
csv_summary = ingest_edgelist(
    edges_csv, nodes_csv, CSV_STORE, (150.0, 150.0, 150.0),
    bin_shape=(37.5, 37.5, 37.5),
    compute_degree=True,
)
print({k: csv_summary[k] for k in ("node_count", "edge_count")})

level = zv.open(CSV_STORE).level(0)
print(f"vertex attrs: {level.attribute_names('vertex')}, link attrs: {level.attribute_names('link')}")
degree = level.read().attributes["degree"]
print(f"mean degree: {degree.mean():.1f}")

node_id,x,y,z,diameter | source,target,length


{'node_count': 3000, 'edge_count': 7428}
vertex attrs: ('degree', 'diameter'), link attrs: ('length',)
mean degree: 5.0


## Summary

| Step | API |
|------|-----|
| Write | `write_graph(path, positions, edges, chunk_shape, bin_shape, kind="graph", vertex_attributes=..., link_attributes=...)` |
| Open | `zarr_vectors.open(path)` → `Dataset` |
| Read all | `read_graph(path)` → `positions`, `edges`, `node_count`, `edge_count` |
| With attributes | `zarr_vectors.open(path).read()` → `positions`, `edges`, `attributes` |
| Bbox query | `read_graph(path, bbox=(lo, hi))` |
| From CSV | `ingest_edgelist(edges_csv, nodes_csv, path, chunk_shape)` |

Graph algorithms (components, search, communities) are in notebooks 07, 08
and 10.